# 04 · RFT training (bf16 LoRA) + eval: Qwen3.6-35B-A3B

**Part 1: Training (Unsloth)**
- bf16 LoRA training starting from the base model. The data comes from two sources:
  - round-1 solutions from `02` (selected in this notebook from `generations.jsonl`)
  - agent episodes from `03` (`agent_sft.jsonl`: math/physics with a Python tool + sandboxed coding)
- The LoRA is saved to Drive and the merged model is written to `/content/merged`.

**Part 2: Eval (vLLM)**
- **Standard eval:** the same settings as the no-tools baseline (4 samples, temperature 0.6, 16k tokens).
  The baseline results are re-scored with the current graders and compared side by side.
- **Tools + agent eval:** the same settings as the `base-tools` eval in `03`, with the same comparison.

Unsloth and vLLM need different torch versions, so **restart the session** between the two parts
(*Runtime → Restart session*). `/content/merged` on disk is kept.
**Do not use "Disconnect and delete runtime"**, or the merged model is deleted.

**Runtime and cost:**
- Runtime: **A100 80GB** (≈6.8 CU/hour).
- Estimate: training 1–3 hours, merge ~15 min, eval ~1 hour. **~20–30 CU** in total.
- `MAX_TRAIN_HOURS` stops training at the time budget. The LoRA trained up to that point is saved and used.

**Before you start:**
- `colab-ai/rft/round1/generations.jsonl` must exist on Drive (written by section B of `02`).
- `colab-ai/rft/agent1/agent_sft.jsonl` from `03` is optional; without it only round-1 data is used.
- `colab-ai/eval/` on Drive must be the current version.

# Part 1: Training

In [ ]:
# Versions are not pinned: Unsloth picks packages that match Colab's torch (Qwen3.5/3.6 need transformers v5)
!pip install -q -U unsloth unsloth_zoo
!pip install -q "transformers==5.2.0"
!pip install -q -U nvidia-nccl-cu13      # torch cu13 builds need a newer NCCL (ncclDevCommDestroy error)
!pip list 2>/dev/null | grep -iE "^(torch|torchvision|triton|xformers|nvidia-nccl-cu1[23]|unsloth|unsloth_zoo|transformers|trl|peft) "

In [ ]:
# Check: do torch + torchvision compiled ops load? If this fails, look at the pip list above
import torch, torchvision.ops
print(torch.__version__, torch.version.cuda, torch.cuda.is_available(), torch.cuda.get_device_name(0))

In [ ]:
MODEL_NAME = "unsloth/Qwen3.6-35B-A3B"
RUN_NAME = "RFT-Agent"
DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
GEN_PATH = f"{DRIVE_ROOT}/rft/round1/generations.jsonl"   # solutions from section B of 02
AGENT_DATA = f"{DRIVE_ROOT}/rft/agent1/agent_sft.jsonl"   # agent episodes selected in 03 (if missing: round 1 only)
OUT_DIR = f"{DRIVE_ROOT}/sft/{RUN_NAME}"
MERGED_DIR = "/content/merged"

# RFT selection
MAX_PER_TASK = 2               # the 2 shortest correct solutions of partially solved problems
EASY_KEEP = 0.5                # fraction of problems solved 4/4 to keep (one solution each)

MAX_SEQ_LENGTH = 10240         # set to 8192 if the memory probe fails; longer examples are skipped
LORA_R = 16
INCLUDE_LINEAR_ATTN = True     # LoRA on the projections of the 30 linear-attention layers as well
TARGET_MLP = False             # LoRA on experts: 256 experts × 40 layers ≈ 1.2B extra params, does not fit in 80 GB

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"   # must be set before torch allocates memory
LEARNING_RATE = 1e-4
NUM_EPOCHS = 2
GRAD_ACCUM = 8
MAX_TRAIN_HOURS = 3.0
HF_REPO = None                 # e.g. "your_username/Qwen3.6-35B-A3B-RFT-Agent-LoRA" (private LoRA upload). None = no upload.
SEED = 3407

In [ ]:
import os, sys, json, glob, time, random, importlib
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount("/content/drive")
login(token=userdata.get("HF_TOKEN"))
os.makedirs(OUT_DIR, exist_ok=True)
assert os.path.exists(GEN_PATH), f"{GEN_PATH} not found: run section B of 02 first"

sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)
importlib.invalidate_caches()
import run_eval as R          # build_messages / tool_calls_of: the same prompt format as generation in 02
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Model + LoRA

In [ ]:
from unsloth import FastModel

model, processor = FastModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit = False,          # QLoRA is not recommended for Qwen3.5/3.6 MoE
    load_in_16bit = True,
    full_finetuning = False,
)
tokenizer = getattr(processor, "tokenizer", processor)

In [ ]:
import torch.nn as nn

# Attention (10 layers) + linear-attention projections (30 layers, names read from the model); experts optional
target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"]
if TARGET_MLP:
    target_modules += ["gate_proj", "up_proj", "down_proj"]
if INCLUDE_LINEAR_ATTN:
    linear_names = sorted({name.rsplit(".", 1)[-1] for name, m in model.named_modules()
                           if ".linear_attn." in name and isinstance(m, nn.Linear)})
    print("linear-attention projections:", linear_names)
    target_modules += [n for n in linear_names if n not in target_modules]

model = FastModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = target_modules,
    lora_alpha = LORA_R,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
    max_seq_length = MAX_SEQ_LENGTH,
)
model.print_trainable_parameters()
import torch
print(f"Model + LoRA GPU memory: {torch.cuda.memory_allocated() / 2**30:.1f} GB (~8+ GB should stay free for activations)")

# Check: which layers got LoRA? (all 40 layers should have something)
layers_with_lora = {int(n.split(".layers.")[1].split(".")[0]) for n, _ in model.named_modules()
                    if "lora_A" in n and ".layers." in n and "visual" not in n}
print(f"Language layers with LoRA: {len(layers_with_lora)}")

## RFT selection + data

- Only **correct and complete** solutions are used, sorted from shortest to longest.
  - From partially solved problems (1/4–3/4), up to `MAX_PER_TASK` solutions are taken. This is where the real learning signal comes from.
  - Problems solved 4/4 are kept at the `EASY_KEEP` rate, with one solution each.
- Examples are rendered with the model's **own chat template**: `reasoning_content` → `<think>`, `tool_calls` → XML.
  This is exactly the format used during generation in `02`.
- Examples longer than `MAX_SEQ_LENGTH` are skipped, not truncated.

In [ ]:
import collections
from datasets import Dataset


def to_example(task, text):
    reasoning, _, answer = text.partition("</think>")
    reasoning, answer = reasoning.strip(), answer.strip()
    msgs = R.build_messages(task)
    if task["type"] == "tool":
        calls = R.tool_calls_of({"content": answer})
        msgs.append({"role": "assistant", "content": answer.split("<tool_call>")[0].strip(),
                     "reasoning_content": reasoning,
                     "tool_calls": [{"type": "function", "function": {"name": c["name"], "arguments": c["arguments"]}}
                                    for c in calls]})
        return {"messages": msgs, "tools": task["tools"]}
    msgs.append({"role": "assistant", "content": answer, "reasoning_content": reasoning})
    return {"messages": msgs}


rng = random.Random(SEED)
rows, picked = [], collections.Counter()
with open(GEN_PATH, encoding="utf-8") as f:
    for line in f:
        row = json.loads(line)
        t, ss = row["task"], row["samples"]
        good = sorted((s for s in ss if s["ok"]), key=lambda s: s["tokens"])
        if not good:
            continue
        if len(good) == len(ss):
            if rng.random() > EASY_KEEP:
                continue
            good = good[:1]
        else:
            good = good[:MAX_PER_TASK]
        src = t["id"].split("-")[0]
        for s in good:
            rows.append({**to_example(t, s["text"]), "source": src})
            picked[src] += 1
if os.path.exists(AGENT_DATA):
    # Multi-turn agent episodes: masking leaves out the tool_response turns (they start with "<|im_start|>user"),
    # every assistant turn is trained
    for l in open(AGENT_DATA, encoding="utf-8"):
        ex = json.loads(l)
        src = "agent_" + ex["source"]
        rows.append({"messages": ex["messages"], "tools": ex["tools"], "source": src})
        picked[src] += 1
print("Selected solutions:", dict(picked), "total", len(rows))

examples, skipped = [], collections.Counter()
for r in rows:
    text = tokenizer.apply_chat_template(r["messages"], tools=r.get("tools"), tokenize=False)
    n = len(tokenizer(text, add_special_tokens=False)["input_ids"])
    if n > MAX_SEQ_LENGTH:
        skipped[r["source"]] += 1
        continue
    examples.append({"text": text, "source": r["source"], "n_tokens": n})

by_src = collections.Counter(e["source"] for e in examples)
print(f"Used: {len(examples)}/{len(rows)}  by source: {dict(by_src)}  skipped (too long): {dict(skipped)}")
print(f"Average length: {sum(e['n_tokens'] for e in examples) / len(examples):.0f} tokens")

train_ds = Dataset.from_list(examples).shuffle(seed=SEED)
print(train_ds[0]["text"][-800:])

## Training

In [ ]:
from transformers import TrainerCallback
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only


class TimeBudget(TrainerCallback):
    # Saves and stops training when the time budget runs out
    def __init__(self, hours):
        self.deadline = time.time() + hours * 3600

    def on_step_end(self, args, state, control, **kwargs):
        if time.time() > self.deadline:
            print("Time budget reached, stopping training.")
            control.should_training_stop = True
            control.should_save = True
        return control


def make_trainer(train, evaluation, **overrides):
    config = dict(
        dataset_text_field = "text",
        max_seq_length = MAX_SEQ_LENGTH,
        per_device_train_batch_size = 1,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio = 0.05,
        num_train_epochs = NUM_EPOCHS,
        learning_rate = LEARNING_RATE,
        lr_scheduler_type = "cosine",
        logging_steps = 5,
        # In-training eval is off: the eval loop keeps all logits (10k tokens × 248k vocab ≈ 17 GB) in memory and OOMs.
        # The real eval is done with vLLM in Part 2.
        eval_strategy = "no",
        save_strategy = "steps",
        save_steps = 25,
        save_total_limit = 2,
        output_dir = f"{OUT_DIR}/checkpoints",
        optim = "adamw_8bit",
        seed = SEED,
        dataset_num_proc = 1,
        report_to = "none",
    )
    config.update(overrides)
    trainer = SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=train, eval_dataset=evaluation,
                         args=SFTConfig(**config))
    # Loss is computed on the assistant response only
    return train_on_responses_only(trainer, instruction_part="<|im_start|>user\n",
                                   response_part="<|im_start|>assistant\n")

In [ ]:
# Memory probe: 2 steps on the 2 longest examples with learning rate 0 (weights do not change).
# If this runs out of memory: Runtime → Restart session, set MAX_SEQ_LENGTH to 8192 in the settings
# and rerun from the settings cell (memory is not fully released after an OOM in the same session).
import torch
longest = Dataset.from_list(sorted(examples, key=lambda e: -e["n_tokens"])[:2])
probe = make_trainer(longest, None, max_steps=2, learning_rate=0.0, gradient_accumulation_steps=1,
                     eval_strategy="no", save_strategy="no", num_train_epochs=1, warmup_ratio=0.0,
                     output_dir="/content/probe")
probe.train()
print(f"Longest example: {longest[0]['n_tokens']} tokens | peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.1f} GB")
del probe

In [ ]:
# Check: is masking right? Only the assistant part should be trained
trainer = make_trainer(train_ds, None)
trainer.add_callback(TimeBudget(MAX_TRAIN_HOURS))
sample = trainer.train_dataset[0]
labels = [t for t in sample["labels"] if t != -100]
print(f"Trained tokens: {len(labels)}/{len(sample['labels'])}")
print(tokenizer.decode(labels)[:400])

In [ ]:
resume = bool(glob.glob(f"{OUT_DIR}/checkpoints/checkpoint-*"))
print("Resume from checkpoint:", resume)
t0 = time.time()
stats = trainer.train(resume_from_checkpoint=resume)
print(stats.metrics)
print(f"Training time: {(time.time() - t0) / 3600:.2f} h ≈ {(time.time() - t0) / 3600 * 6.77:.1f} CU")

## Save + merge

In [ ]:
model.save_pretrained(f"{OUT_DIR}/lora")
tokenizer.save_pretrained(f"{OUT_DIR}/lora")
if HF_REPO:
    model.push_to_hub(HF_REPO, private=True)
    tokenizer.push_to_hub(HF_REPO, private=True)

# Merge the LoRA into the weights for the vLLM eval (bf16, ~70 GB, local disk)
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")

# Tokenizer + chat template always come from the original base model (the LoRA does not change them; the tokenizer
# rewritten by transformers v5 can split text differently). Missing helper files (preprocessor config etc.) are copied too.
import shutil
from huggingface_hub import snapshot_download
base = snapshot_download("Qwen/Qwen3.6-35B-A3B-FP8", allow_patterns=["*.json", "*.jinja", "*.txt"])
TOKENIZER_FILES = {"tokenizer.json", "tokenizer_config.json", "vocab.json", "merges.txt", "chat_template.jinja",
                   "added_tokens.json", "special_tokens_map.json"}
for f in os.listdir(base):
    if f.endswith(".safetensors.index.json") or f in ("config.json", "generation_config.json"):
        continue
    if f in TOKENIZER_FILES or not os.path.exists(f"{MERGED_DIR}/{f}"):
        shutil.copy(f"{base}/{f}", MERGED_DIR)
        print("copied:", f)
print("architectures:", json.load(open(f"{MERGED_DIR}/config.json"))["architectures"])
!du -sh {MERGED_DIR}

In [ ]:
# Install vLLM for Part 2 (this changes the torch version; restart the session afterwards)
!pip install -q -U vllm pytest

# Part 2: Eval

**Now:** *Runtime → Restart session*, then continue with the cells below.
**Do not use "Disconnect and delete runtime"**: `/content/merged` would be deleted.

In [ ]:
RUN_NAME = "RFT-Agent"
DRIVE_ROOT = "/content/drive/MyDrive/colab-ai"
EVAL_DIR = f"{DRIVE_ROOT}/eval"
MERGED_DIR = "/content/merged"
LABEL = f"{RUN_NAME}-colab-fp8"
BASE_LABEL = "base-colab-fp8"
CU_PER_HOUR = 6.77
SEED = 3407

# Same settings as the no-tools baseline
EVAL_SETS = ["personal", "math500", "aime2025", "gpqa_physics"]
EVAL_SAMPLES = 4
EVAL_TEMPERATURE = 0.6
EVAL_MAX_TOKENS = 16384
MAX_MODEL_LEN = 20480

# Tools + agent eval: same settings as the base-tools eval in 03
RUN_TOOLS_EVAL = True
TOOLS_LABEL = f"{RUN_NAME}-tools16k-colab-fp8"
BASE_TOOLS_LABEL = "base-tools16k-colab-fp8"
TOOLS_SETS = ["math500", "aime2025", "gpqa_physics"]
TOOLS_SAMPLES = 2
TOOLS_CONTEXT, TOOLS_GEN_TOKENS, TOOLS_TURN_TOKENS = 20480, 16384, 16384
MAX_TURNS = {"python": 6, "code": 10}

In [ ]:
import os, sys, json, glob, time, importlib, collections
from datetime import datetime
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from google.colab import drive

drive.mount("/content/drive")
sys.path.insert(0, EVAL_DIR)
sys.path_importer_cache.pop(EVAL_DIR, None)
importlib.invalidate_caches()
import run_eval as R
import agent as A
assert os.path.exists(f"{MERGED_DIR}/config.json"), "No merged model: Part 1 must finish (without 'Disconnect and delete runtime')"

In [ ]:
from vllm import LLM, SamplingParams

llm = LLM(
    model = MERGED_DIR,
    quantization = "fp8",          # same condition as the base model: FP8 weights (Marlin on A100)
    max_model_len = MAX_MODEL_LEN,
    gpu_memory_utilization = 0.92,
    limit_mm_per_prompt = {"image": 0, "video": 0},
    enable_prefix_caching = True,
    seed = SEED,
)
tok = llm.get_tokenizer()


def to_prompt(task):
    return tok.apply_chat_template(R.build_messages(task), tools=task.get("tools"),
                                   tokenize=False, add_generation_prompt=True)


def generate(tasks, n, temperature, max_tokens):
    params = SamplingParams(n=n, temperature=temperature, top_p=0.95, top_k=20,
                            max_tokens=max_tokens, seed=SEED)
    outs = llm.generate([to_prompt(t) for t in tasks], params)
    samples = [[{"text": o.text, "tokens": len(o.token_ids), "finish": o.finish_reason} for o in out.outputs]
               for out in outs]
    flat = [(t, s) for t, ss in zip(tasks, samples) for s in ss]
    with ThreadPoolExecutor(max_workers=(os.cpu_count() or 4) * 2) as ex:
        results = list(ex.map(lambda ts: R.grade(ts[0], {"content": ts[1]["text"]}), flat))
    for (t, s), (ok, detail, _) in zip(flat, results):
        s["ok"] = bool(ok) and s["finish"] == "stop"
        s["detail"] = str(detail)[:300]
    return samples


print(generate([{"id": "t", "type": "math", "prompt": "What is 2+2?", "answer": "4"}], 1, 0.6, 2048)[0][0])

In [ ]:
eval_files = [f"{EVAL_DIR}/tasks/{n}.jsonl" for n in EVAL_SETS if os.path.exists(f"{EVAL_DIR}/tasks/{n}.jsonl")]
eval_tasks = R.load_tasks(eval_files, 0)
print(f"{len(eval_tasks)} tasks × {EVAL_SAMPLES} samples")

t0 = time.time()
samples = generate(eval_tasks, EVAL_SAMPLES, EVAL_TEMPERATURE, EVAL_MAX_TOKENS)
elapsed = time.time() - t0

stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
results_dir = f"{EVAL_DIR}/results"
new_path = f"{results_dir}/{LABEL}_{stamp}.jsonl"
stats = {}
with open(new_path, "w", encoding="utf-8") as f:
    for t, ss in zip(eval_tasks, samples):
        st = stats.setdefault(t["_source"], {"n": 0, "correct": 0, "tokens": 0, "seconds": 0.0})
        for s in ss:
            st["n"] += 1
            st["correct"] += int(s["ok"])
            st["tokens"] += s["tokens"]
            f.write(json.dumps({"id": t["id"], "source": t["_source"], "type": t["type"], "ok": s["ok"],
                                "detail": s["detail"], "tokens": s["tokens"], "finish": s["finish"],
                                "response": R.strip_thinking(s["text"])}, ensure_ascii=False) + "\n")
total = sum(st["n"] for st in stats.values())
for st in stats.values():
    st["seconds"] = elapsed * st["n"] / total
R.write_summary(results_dir, stamp, LABEL, MERGED_DIR, stats)
print(f"Time: {elapsed / 60:.1f} min ≈ {elapsed / 3600 * CU_PER_HOUR:.1f} CU")

## Comparison: base vs RFT

Both files are re-scored with **the same current graders**; truncated solutions count as wrong.
**Gate:** if the RFT model does not beat the base model, it is not used.

In [ ]:
tasks = {}
for p in glob.glob(f"{EVAL_DIR}/tasks/*.jsonl"):
    for l in open(p, encoding="utf-8"):
        if l.strip():
            t = json.loads(l)
            tasks[(Path(p).stem, t["id"])] = t


def score_file(path):
    st = collections.defaultdict(collections.Counter)
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        ok = r["finish"] == "stop" and R.grade(tasks[(r["source"], r["id"])], {"content": r["response"]})[0]
        s = st[r["source"]]
        s["n"] += 1
        s["ok"] += ok
        s["truncated"] += r["finish"] == "length"
        s["tok"] += r["tokens"]
    return st


base_path = sorted(glob.glob(f"{EVAL_DIR}/results/{BASE_LABEL}_*.jsonl"))[-1]
base, new = score_file(base_path), score_file(new_path)
print(f"{'set':<13} {'base':>7} {'RFT':>7} {'diff':>7} | {'trunc B':>9} {'trunc R':>9} | {'tok B':>6} {'tok R':>6}")
for src in base:
    b, n = base[src], new[src]
    ab, an = b["ok"] / b["n"], n["ok"] / n["n"]
    print(f"{src:<13} {ab:>7.1%} {an:>7.1%} {an - ab:>+7.1%} | {b['truncated'] / b['n']:>9.1%} {n['truncated'] / n['n']:>9.1%} | "
          f"{b['tok'] // b['n']:>6} {n['tok'] // n['n']:>6}")

## Tools + agent eval

The same agent loop as in `03`:
- MATH-500, AIME and GPQA physics problems are solved with the Python tool.
- The held-out set of 100 agentic coding tasks (`agent_code.jsonl`) is evaluated as well.

The result is compared with the `base-tools` eval: accuracy, tokens, turns and tool calls.

In [ ]:
WORKERS = (os.cpu_count() or 4) * 2


def run_episodes(episodes, temperature, context_limit, gen_limit, turn_limit, label=""):
    # Same as 03: each turn generates all active episodes in one batch, tools run in parallel
    active = [e for e in episodes if not e.done]
    while active:
        prompts, params, batch = [], [], []
        for e in active:
            p = tok.apply_chat_template(e.messages, tools=e.tools, tokenize=False, add_generation_prompt=True)
            budget = min(turn_limit, context_limit - len(tok(p, add_special_tokens=False).input_ids),
                         gen_limit - e.tokens)
            if budget < 256:
                e.fail("context")
                continue
            prompts.append(p)
            params.append(SamplingParams(temperature=temperature, top_p=0.95, top_k=20, max_tokens=budget,
                                         seed=SEED + 7919 * e.rollout + e.turns))
            batch.append(e)
        if not batch:
            break
        print(f"{label} turn {batch[0].turns + 1}: {len(batch)} active episodes")
        outs = llm.generate(prompts, params, use_tqdm=True)
        with ThreadPoolExecutor(max_workers=WORKERS) as ex:
            list(ex.map(lambda eo: eo[0].consume(eo[1].outputs[0].text, eo[1].outputs[0].finish_reason,
                                                  len(eo[1].outputs[0].token_ids)), zip(batch, outs)))
        active = [e for e in batch if not e.done]
    with ThreadPoolExecutor(max_workers=WORKERS) as ex:
        results = list(ex.map(A.grade_episode, episodes))
    for e, (ok, detail) in zip(episodes, results):
        e.ok, e.detail = ok, str(detail)[:300]
        e.sandbox.close()
    return episodes


if RUN_TOOLS_EVAL:
    tool_tasks = R.load_tasks([f"{EVAL_DIR}/tasks/{n}.jsonl" for n in TOOLS_SETS], 0)
    for t in tool_tasks:
        t["_source"] += "+py"
    agent_tasks = R.load_tasks([f"{EVAL_DIR}/tasks/agent_code.jsonl"], 0)
    for t in agent_tasks:
        t["_source"] = "agent_code"
    episodes = [A.Episode(t, rollout=k, max_turns=MAX_TURNS["code" if t["type"] == "agent_code" else "python"])
                for t in tool_tasks + agent_tasks for k in range(TOOLS_SAMPLES)]
    t0 = time.time()
    run_episodes(episodes, EVAL_TEMPERATURE, TOOLS_CONTEXT, TOOLS_GEN_TOKENS, TOOLS_TURN_TOKENS, "eval")
    elapsed = time.time() - t0
    stamp = datetime.now().strftime("%Y%m%d-%H%M%S")
    tools_path = f"{EVAL_DIR}/results/{TOOLS_LABEL}_{stamp}.jsonl"
    stats = {}
    with open(tools_path, "w", encoding="utf-8") as f:
        for e in episodes:
            st = stats.setdefault(e.task["_source"], {"n": 0, "correct": 0, "tokens": 0, "seconds": 0.0})
            st["n"] += 1
            st["correct"] += int(e.ok)
            st["tokens"] += e.tokens
            f.write(json.dumps({"id": e.task["id"], "source": e.task["_source"], "ok": e.ok, "detail": e.detail,
                                "tokens": e.tokens, "turns": e.turns, "tool_calls": e.n_tool_calls,
                                "failed": e.failed}, ensure_ascii=False) + "\n")
    total = sum(st["n"] for st in stats.values())
    for st in stats.values():
        st["seconds"] = elapsed * st["n"] / total
    R.write_summary(f"{EVAL_DIR}/results", stamp, TOOLS_LABEL, MERGED_DIR, stats)
    print(f"Time: {elapsed / 60:.1f} min ≈ {elapsed / 3600 * CU_PER_HOUR:.1f} CU")

In [ ]:
def tools_stats(path):
    st = collections.defaultdict(collections.Counter)
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        s = st[r["source"]]
        s["n"] += 1
        s["ok"] += r["ok"]
        s["tok"] += r["tokens"]
        s["turns"] += r["turns"]
        s["tools"] += r["tool_calls"]
        s["unfinished"] += r["failed"] in ("length", "context", "turns")
    return st


base_tools = tools_stats(sorted(glob.glob(f"{EVAL_DIR}/results/{BASE_TOOLS_LABEL}_*.jsonl"))[-1])
new_tools = tools_stats(tools_path)
print(f"{'set':<17} {'base':>7} {'RFT':>7} {'diff':>7} | {'unfin B':>10} {'unfin R':>10} | "
      f"{'tok B':>6} {'tok R':>6} | {'tools B':>7} {'tools R':>7}")
for src in base_tools:
    b, n = base_tools[src], new_tools[src]
    ab, an = b["ok"] / b["n"], n["ok"] / n["n"]
    print(f"{src:<17} {ab:>7.1%} {an:>7.1%} {an - ab:>+7.1%} | {b['unfinished'] / b['n']:>10.1%} {n['unfinished'] / n['n']:>10.1%} | "
          f"{b['tok'] // b['n']:>6} {n['tok'] // n['n']:>6} | {b['tools'] / b['n']:>7.1f} {n['tools'] / n['n']:>7.1f}")

## Finish: stop spending CU

In [ ]:
from google.colab import runtime
runtime.unassign()